In [12]:
from dotenv import load_dotenv    
import os                         

load_dotenv()                    

from langchain_openai import ChatOpenAI                          
from langchain_core.messages import HumanMessage, SystemMessage  
from langgraph.graph import StateGraph, START, END               
from langgraph.graph.message import add_messages                 
from typing import TypedDict, Annotated
from langchain_core.tools import tool
import requests
from bs4 import BeautifulSoup



In [13]:
class State(TypedDict):
    messages: Annotated[list, add_messages]


In [14]:
llm = ChatOpenAI(
    model="gpt-4o-mini",                              
    api_key=os.getenv("API_TOKEN"),                    
    base_url="https://openrouter.ai/api/v1"            
)

In [15]:
@tool
def scrape_website(url: str) -> str:
    """Scrape a publicly accessible website and return its title,
    headings, paragraphs, and links.

    Args:
        url: HTTPS URL of the webpage to scrape.
    """
    try:
        parsed = urlparse(url)

        if parsed.scheme != "https" or not parsed.hostname:
            return "Error: Please provide a valid HTTPS URL."

        response = requests.get(
            url,
            headers={"User-Agent": "Mozilla/5.0"},
            timeout=20,
            allow_redirects=False,
        )
        response.raise_for_status()

        content_type = response.headers.get("Content-Type", "")
        if "text/html" not in content_type.lower():
            return "Error: URL does not contain an HTML page."

        soup = BeautifulSoup(response.text, "html.parser")

        for element in soup(
            ["script", "style", "noscript", "svg"]
        ):
            element.decompose()

        title = (
            soup.title.get_text(strip=True)
            if soup.title else "No title found"
        )

        headings = [
            h.get_text(" ", strip=True)
            for h in soup.find_all(["h1", "h2", "h3"])
        ]

        paragraphs = [
            p.get_text(" ", strip=True)
            for p in soup.find_all("p")
            if p.get_text(strip=True)
        ]

        links = [
            f"{a.get_text(' ', strip=True)}: "
            f"{urljoin(url, a['href'])}"
            for a in soup.find_all("a", href=True)
        ]

        return (
            f"URL: {url}\n"
            f"TITLE: {title}\n\n"
            f"HEADINGS:\n" + "\n".join(headings) +
            "\n\nPARAGRAPHS:\n" + "\n".join(paragraphs) +
            "\n\nLINKS:\n" + "\n".join(links)
        )

    except requests.RequestException as e:
        return f"Website scraping error: {e}"
    except Exception as e:
        return f"Error: {e}"


    


In [16]:
tools = [scrape_website]

In [22]:
llm_with_tools = llm.bind_tools(tools)

In [17]:
def chatbot(state: State) -> dict:
    """
    The chatbot node. Takes the current messages,
    sends them to the LLM, and returns the response.
    """
    
    system = SystemMessage(content="You are a helpful and friendly assistant.")
    
    
    response = llm.invoke([system] + state["messages"])
    
    
    return {"messages": [response]}


In [18]:
graph_builder = StateGraph(State)              

graph_builder.add_node("chatbot", chatbot)     

graph_builder.add_edge(START, "chatbot")       
graph_builder.add_edge("chatbot", END)        


In [19]:
graph = graph_builder.compile()


In [20]:
result = graph.invoke({
    "messages": [HumanMessage(content="Who is the president of Nigeria?")]
})


In [21]:
print(result["messages"][-1].content)


As of my last knowledge update in October 2023, the President of Nigeria is Bola Ahmed Tinubu. He took office on May 29, 2023. However, please verify with up-to-date sources, as political situations can change.
